# 06 GNN Evaluation Synthesis

This notebook combines the message-passing ablation, GraphSAGE-versus-GCN comparison, and configuration-matched RF/XGBoost baselines. It contains no additional model training.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [_cwd, *_cwd.parents]
        if (candidate / "00_data_graph").exists()
        and (candidate / "04_supervised").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the dissertation workspace.")

GNN_RESULTS = PROJECT_ROOT / '04_supervised' / '02_gnn_models' / 'graphsage_gcn_multiseed_results'
EVAL_RESULTS = PROJECT_ROOT / '04_supervised' / '03_gnn_evaluation' / 'message_passing_and_baseline_results'
required = [
    GNN_RESULTS / 'multiseed_summary.csv',
    GNN_RESULTS / 'multiseed_graphsage_vs_gcn.csv',
    EVAL_RESULTS / 'p3_ablation_primary_comparisons.csv',
    EVAL_RESULTS / 'symmetric_multiseed_comparison.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Run notebooks 03, 04, and 05 first. Missing: {missing}')

## GraphSAGE and GCN

In [2]:
gnn_summary = pd.read_csv(GNN_RESULTS / 'multiseed_summary.csv')
gnn_comparison = pd.read_csv(GNN_RESULTS / 'multiseed_graphsage_vs_gcn.csv')
display(gnn_summary)
display(gnn_comparison)

,condition,n_seeds,macro_f1_mean,macro_f1_std
0,GraphSAGE,5,0.8274,0.0217
1,GCN,5,0.8255,0.0238


,comparison,n_seeds,mean_diff,diff_95ci_lo,diff_95ci_hi,paired_t_stat,paired_p_value,pct_seeds_graphsage_better
0,GraphSAGE - GCN (both structural+transaction_b...,5,0.0019,-0.0158,0.0196,0.3009,0.7785,0.6


## Message-passing and topology ablation

In [3]:
p3_comparison = pd.read_csv(EVAL_RESULTS / 'p3_ablation_primary_comparisons.csv')
display(p3_comparison)

,comparison,n_seeds,mean_diff,diff_95ci_lo,diff_95ci_hi,paired_t_stat,paired_p_value,pct_seeds_a_better
0,GraphSAGE(real edges) - MLP(no edges),5,0.0339,-0.0163,0.0840,1.8740,0.1342,0.8
1,GraphSAGE(real edges) - GraphSAGE(degree-prese...,5,0.1455,0.0228,0.2682,3.2919,0.0302,1.0


## Matched tree baselines

In [4]:
baseline_comparison = pd.read_csv(EVAL_RESULTS / 'symmetric_multiseed_comparison.csv')
display(baseline_comparison)

,gnn,baseline,n_seeds,gnn_mean,baseline_mean,mean_diff,diff_95ci_lo,diff_95ci_hi,paired_t_stat,paired_p_value,pct_seeds_gnn_better
0,GraphSAGE,RandomForest_graph_structural,5,0.8274,0.7860,0.0414,0.0115,0.0713,3.8472,0.0183,1.0
1,GraphSAGE,XGBoost_graph_structural,5,0.8274,0.8146,0.0128,-0.0174,0.0429,1.1743,0.3054,0.6
2,GraphSAGE,RandomForest_hand_crafted,5,0.8274,0.9084,-0.0810,-0.1110,-0.0510,-7.4908,0.0017,0.0
3,GraphSAGE,XGBoost_hand_crafted,5,0.8274,0.8843,-0.0569,-0.0870,-0.0267,-5.2349,0.0064,0.0
4,GCN,RandomForest_graph_structural,5,0.8255,0.7860,0.0395,0.0051,0.0739,3.1865,0.0333,1.0
5,GCN,XGBoost_graph_structural,5,0.8255,0.8146,0.0108,-0.0223,0.0439,0.9094,0.4146,0.4
6,GCN,RandomForest_hand_crafted,5,0.8255,0.9084,-0.0829,-0.1136,-0.0522,-7.5043,0.0017,0.0
7,GCN,XGBoost_hand_crafted,5,0.8255,0.8843,-0.0588,-0.0919,-0.0257,-4.9307,0.0079,0.0


## Thesis-facing interpretation

GraphSAGE and GCN show near-parity under the matched multi-seed design. Real transaction topology outperforms degree-preserving rewired topology in the P3 ablation. Both GNNs remain below Random Forest and XGBoost supplied with the complete hand-crafted feature set. These conclusions apply to the fixed, transductive observed-graph design.